<a href="https://colab.research.google.com/github/Liao-HsienTing/PL-Repo./blob/main/114_1_HW6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, time, uuid, re, json
import pandas as pd
import gradio as gr
from io import StringIO # 用於在記憶體中讀取 CSV
from datetime import datetime as dt, timedelta
from dateutil.tz import gettz
import pytz
from IPython.display import display, Markdown

import google.generativeai as genai
from google.colab import auth, files, userdata
import gspread
from gspread_dataframe import set_with_dataframe, get_as_dataframe
from google.auth import default

In [2]:
# --- 1. Google Colab 相關授權 ---
print("正在進行 Google 帳號授權...")
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("✅ Google 帳號授權完成。")

正在進行 Google 帳號授權...
✅ Google 帳號授權完成。


In [3]:
# --- 2. 設定 Gemini API 金鑰 ---
try:
    # 從 Colab Secrets 中獲取 API 金鑰
    api_key = userdata.get('GOOGLE_API_KEY')
    if api_key is None:
        raise ValueError("找不到 'GOOGLE_API_KEY' API 金鑰，請檢查 Colab Secrets。")
    genai.configure(api_key=api_key)
    print("✅ Gemini API 金鑰設定完成。")
except Exception as e:
    print(f"❌ Gemini API 金鑰設定失敗：{e}")

✅ Gemini API 金鑰設定完成。


In [4]:
# --- 3. 定義共用的 Google Sheet 資訊 ---
# 這是您指定的統一 URL
SHEET_URL = "https://docs.google.com/spreadsheets/d/1JuCgkGhUJ24wFWibuJxBLrCXV1qAtIsCISFsvdk0jHs/edit?usp=sharing"
print(f"將使用此 Google Sheet: {SHEET_URL}")

將使用此 Google Sheet: https://docs.google.com/spreadsheets/d/1JuCgkGhUJ24wFWibuJxBLrCXV1qAtIsCISFsvdk0jHs/edit?usp=sharing


In [5]:
print("請上傳您的 PDF 課表檔案...")
# 顯示上傳按鈕
uploaded = files.upload()

if not uploaded:
    print("\n⚠️ 您沒有上傳任何檔案。")
else:
    # 取得上傳檔案的名稱
    pdf_filename = next(iter(uploaded))
    print(f"\n✅ 成功上傳檔案: '{pdf_filename}'")

    # 檢查是否為 PDF (簡易檢查)
    if not pdf_filename.lower().endswith('.pdf'):
        print(f"⚠️ 警告：您上傳的檔案 '{pdf_filename}' 不是 PDF 檔。")

    # 將檔案名稱存到 Colab 的暫存記憶體中，供下個區塊使用
    %store pdf_filename

請上傳您的 PDF 課表檔案...


Saving export.pdf to export.pdf

✅ 成功上傳檔案: 'export.pdf'
Stored 'pdf_filename' (str)


In [14]:
# --- 1. 定義 Google Sheet 目標 ---
WORKSHEET_NAME = "HW6"
gemini_file = None # 在 try 之前先宣告變數

try:
    # --- 2. 讀取上一個儲存格存的檔案名稱 ---
    %store -r pdf_filename
    if not os.path.exists(pdf_filename):
        print(f"❌ 找不到檔案 '{pdf_filename}'，請重新執行上一個儲存格上傳。")
    else:
        print(f"準備處理檔案: '{pdf_filename}'")

        # --- 3. 上傳檔案到 Gemini File API ---
        print("正在上傳 PDF 至 Gemini File API...")
        gemini_file = genai.upload_file(path=pdf_filename,
                                        display_name=f"課表PDF-{pdf_filename}")
        print(f"✅ 檔案上傳成功！(File URI: {gemini_file.uri})")

        # --- 4. 準備 Prompt ---
        prompt_text = f"""
        你是一位 helpful AI assistant。
        請分析我上傳的 PDF 課表檔案 (檔案名稱: {pdf_filename})。

        這份 PDF 課表是「國立臺灣師範大學114學年度第1學期課表」。
        授課教師是「蔡芸琤」。

        請幫我將這份課表，轉換成每周的紀錄。
        學期開學日是 2025/9/1，學期結束日是 2025/12/19。

        你的輸出**必須**是 CSV 格式，並且**只**輸出 CSV 內容 (包含標頭)，
        不要有任何其他說明文字 (例如 "好的，這是..." 或 "```csv" 標記)。

        CSV 格式的欄位必須如下：
        "日期","星期","課程名稱","時間(起)","時間(迄)","地點","攜帶品","先讀章節","備註"

        - 「攜帶品」、「先讀章節」、「備註」這三欄因為 PDF 中沒有，請保持空白。
        - 「日期」範圍請涵蓋 2025/9/1 到 2025/12/19 之間所有有課的日期。
        - 「請仔細解析 PDF 中的表格，找出所有課程、時間和地點。
        """

        # --- 5. 呼叫 Gemini 模型 ---
        print("正在向 Gemini 2.5 Pro 提出請求 (這可能需要一點時間)...")
        model = genai.GenerativeModel('gemini-2.5-pro')
        response = model.generate_content([prompt_text, gemini_file])

        # --- 6. 顯示 Gemini 的「原始」輸出 (Debug) ---
        print("\n--- 🤖 Gemini 原始輸出 (Debug) ---")
        print(response.text)
        print("---------------------------------\n")

        # --- 7. 強化 CSV 內容的清理 ---
        csv_output = response.text.strip().replace("```csv\n", "").replace("```", "")

        # --- 8. (*** 修正點 ***) 將 CSV 讀入 DataFrame 並 *立即* 檢查 ---
        print("正在將 Gemini 輸出解析為 DataFrame...")
        try:
            # 讓 pandas 直接嘗試讀取
            df_to_upload = pd.read_csv(StringIO(csv_output))
        except Exception as parse_e:
            # 如果連 pandas 都無法解析，那才是真的格式錯誤
            print(f"❌ Pandas 無法解析 Gemini 輸出的 CSV: {parse_e}")
            raise ValueError("Gemini 輸出的內容不是有效的 CSV 格式。")

        # (新) 檢查 DataFrame 的標頭是否正確
        if df_to_upload.empty or df_to_upload.columns[0] != "日期":
            print(f"❌ 解析後的 DataFrame 標頭不符。偵測到的標頭為: {df_to_upload.columns}")
            raise ValueError("Gemini 輸出的 CSV 標頭不符，第一個欄位應為 '日期'。")

        # --- 9. 格式化並上傳 ---
        df_to_upload = df_to_upload.fillna('')
        print(f"✅ 成功將 Gemini 內容解析為 DataFrame，共 {len(df_to_upload)} 筆資料。")

        print(f"正在開啟 Google Sheet (URL: {SHEET_URL})...")
        gsheets = gc.open_by_url(SHEET_URL)

        print(f"正在存取分頁: '{WORKSHEET_NAME}'...")
        try:
            worksheet = gsheets.worksheet(WORKSHEET_NAME)
        except gspread.exceptions.WorksheetNotFound:
            print(f"找不到分頁 '{WORKSHEET_NAME}'，將自動建立新分頁...")
            worksheet = gsheets.add_worksheet(title=WORKSHEET_NAME, rows="100", cols="20")

        print("正在清空分頁內容...")
        worksheet.clear()

        print(f"正在將 {len(df_to_upload)} 筆資料寫入 '{WORKSHEET_NAME}' 分頁...")
        set_with_dataframe(worksheet, df_to_upload)

        print(f"\n✅ 成功將 Gemini 產生的課表寫入至 '{WORKSHEET_NAME}' 分頁！")

except NameError:
    print("❌ 找不到 'pdf_filename'。請先執行 [區塊 2] 來上傳檔案。")
except Exception as e:
    print(f"❌ 執行時發生錯誤：{e}")

finally:
    # --- 10. 最終清理 ---
    if gemini_file:
        try:
            print("正在從 Gemini 雲端刪除暫存檔案 (Finally)...")
            genai.delete_file(gemini_file.name)
            print("✅ 暫存檔案已刪除。")
        except Exception as del_e:
            print(f"⚠️ 清理檔案時發生非致命錯誤 (可能已刪除): {del_e}")

準備處理檔案: 'export.pdf'
正在上傳 PDF 至 Gemini File API...
✅ 檔案上傳成功！(File URI: https://generativelanguage.googleapis.com/v1beta/files/9ahpjxlrfxfy)
正在向 Gemini 2.5 Pro 提出請求 (這可能需要一點時間)...

--- 🤖 Gemini 原始輸出 (Debug) ---
"日期","星期","課程名稱","時間(起)","時間(迄)","地點","攜帶品","先讀章節","備註"
"2025/9/1","星期一","全民國防教育軍事訓練-國防科技","08:10","10:00","誠201","","",""
"2025/9/1","星期一","輔導原理與實務(教)","10:20","12:10","樸301","","",""
"2025/9/1","星期一","班級經營(教)","15:30","17:20","誠107","","",""
"2025/9/2","星期二","木工製造","08:10","12:10","科技系TC501教室","","",""
"2025/9/2","星期二","色彩學","13:20","16:20","科技系TB213教室","","",""
"2025/9/2","星期二","展示設計","17:30","20:25","科技系TB213教室","","",""
"2025/9/3","星期三","運算思維與程式設計","15:30","17:20","教401","","",""
"2025/9/4","星期四","程式語言","09:10","12:10","科技系TB311教室","","",""
"2025/9/4","星期四","教育概論(教)","13:20","15:10","教312","","",""
"2025/9/4","星期四","機構設計","17:30","20:25","科技系TC504教室","","",""
"2025/9/5","星期五","科技系統與社會發展","10:20","12:10","科技系TB311教室","","",""
"2025/9/5","星期五","電腦輔助製圖","15:30","18:20","科技系TA50

In [28]:
# === 區塊 5 (最終版)：讀取「課程內容」的 AI 助理 ===

# === 1. 參數設定 (全部集中於此) ===
# (確保區塊 1 的所有套件都已載入)
WORKSHEET_NAME = "HW6"
SHEET_WEEKLY = "本週重點" # 儲存歷史紀錄的分頁
TIMEZONE = "Asia/Taipei"

# === 2. 小工具函式 (Helper Functions) ===
# (與您前一版相同)
TW_TZ = pytz.timezone(TIMEZONE)
week_map_tw = {"一":0,"二":1,"三":2,"四":3,"五":4}
week_map_en = {"Mon":0,"Tue":1,"Wed":2,"Thu":3,"Fri":4}

def normalize_weekday(w):
    if w in week_map_tw: return week_map_tw[w]
    if w in week_map_en: return week_map_en[w]
    raise ValueError("星期請用：一~日 或 Mon~Sun")

def ensure_datetime(s):
    try: return pd.to_datetime(s).date()
    except Exception: return pd.NaT

def week_monday(any_date):
    return any_date - timedelta(days=any_date.weekday())

def date_range_this_week(today=None):
    now = dt.now(TW_TZ).date() if today is None else today
    mon = week_monday(now)
    sun = mon + timedelta(days=6)
    return mon, sun

def summarize_courses(day_df):
    if day_df.empty: return "本日無課"
    parts = []
    for _, r in day_df.iterrows():
        parts.append(f"{r['時間(起)']}-{r['時間(迄)']} {r['課程名稱']}（{r['地點']}）")
    return "；".join(parts)

def make_one_line_reminder(rows):
    if rows.empty:
        return "本日無課，不需準備。"
    items = []
    readings = []
    for _, r in rows.iterrows():
        if str(r.get("攜帶品","")).strip():
            items.append(str(r["攜帶品"]).strip())
        if str(r.get("先讀章節","")).strip():
            readings.append(str(r["先讀章節"]).strip())
    items_txt = "；".join(dict.fromkeys(items)) if items else "一般上課用品"
    read_txt  = "；".join(dict.fromkeys(readings)) if readings else "無指定預習"
    return f"請攜帶：{items_txt}；預習：{read_txt}。"

# === 3. Gradio 核心函式 ===

# (全域載入課表與 Gemini 模型)
print(f"正在從 Google Sheet '{WORKSHEET_NAME}' 預載課表資料...")
try:
    gsheets = gc.open_by_url(SHEET_URL)
    sheets_values = gsheets.worksheet(WORKSHEET_NAME).get_all_values()
    GLOBAL_DF = pd.DataFrame(sheets_values[1:], columns=sheets_values[0])
    GLOBAL_DF["日期"] = GLOBAL_DF["日期"].apply(ensure_datetime)
    GLOBAL_DF = GLOBAL_DF.dropna(subset=["日期"])
    # (新) 確保所有欄位都是字串，避免 None 造成錯誤
    GLOBAL_DF = GLOBAL_DF.fillna("")
    print(f"✅ 成功預載 {len(GLOBAL_DF)} 筆課表資料。")
except Exception as e:
    print(f"❌ 預載課表時發生錯誤：{e}")
    GLOBAL_DF = pd.DataFrame()

try:
    GEMINI_MODEL = genai.GenerativeModel('gemini-2.5-flash')
    print("✅ 成功初始化 Gemini Flash 模型。")
except Exception as e:
    print(f"❌ 初始化 Gemini 模型失敗：{e}")
    GEMINI_MODEL = None

def query_schedule_and_reminder(selected_weekday):
    """
    Gradio 按下「查詢」按鈕時執行的主函式
    """
    if GLOBAL_DF.empty:
        return "❌ 錯誤：課表資料未載入", pd.DataFrame(), "請檢查 Colab 輸出", {}, gr.update(visible=False), gr.update(visible=False), " "

    # 1. 計算日期範圍
    target_wd = normalize_weekday(selected_weekday)
    this_mon, this_sun = date_range_this_week()

    # 2. 篩選資料
    mask_week = (GLOBAL_DF["日期"] >= this_mon) & (GLOBAL_DF["日期"] <= this_sun)
    mask_wd   = GLOBAL_DF["日期"].apply(lambda d: d.weekday()==target_wd)
    today_df  = GLOBAL_DF[mask_week & mask_wd].copy().sort_values(["日期","時間(起)"])

    # 3. 產生提醒
    one_line = make_one_line_reminder(today_df)

    # 4. 準備要顯示的標題和 DataFrame
    title = f"### ✅ 本週（{this_mon} ~ {this_sun}） 星期 {selected_weekday} 課程"
    # (新) 顯示出「備註」和「先讀章節」，讓使用者知道 AI 讀到了什麼
    display_cols = ["課程名稱","時間(起)","時間(迄)","地點","先讀章節","備註"]
    display_df = today_df[display_cols] if not today_df.empty else pd.DataFrame(columns=display_cols)

    # 5. 準備要「隱藏」的狀態資料
    state_data = {
        "this_mon": this_mon.isoformat(),
        "this_sun": this_sun.isoformat(),
        "target_weekday": selected_weekday,
        # (新) 將 today_df 存入 state，AI 函式才能讀取
        "today_df_json": today_df.to_json(orient='split', date_format='iso'),
    }

    # 6. 準備 AI 推薦區的下拉選單
    course_list = today_df["課程名稱"].unique().tolist()
    if course_list:
        dropdown_update = gr.update(choices=course_list, value=course_list[0], visible=True)
        button_update = gr.update(visible=True)
        ai_title_update = gr.update(visible=True)
    else:
        dropdown_update = gr.update(choices=[], visible=False)
        button_update = gr.update(visible=False)
        ai_title_update = gr.update(visible=False)

    # 7. 清空上一次的 AI 推薦結果
    resources_output = " "

    return title, display_df, one_line, state_data, dropdown_update, button_update, ai_title_update, resources_output

def save_edited_reminder(edited_text, current_state):
    """
    Gradio 按下「儲存」按鈕時執行的函式
    """
    # (此函式與前一版相同)
    if not current_state:
        return "❌ 錯誤：請先「查詢課表」後再儲存。"
    try:
        this_mon = dt.fromisoformat(current_state["this_mon"]).date()
        this_sun = dt.fromisoformat(current_state["this_sun"]).date()
        target_weekday = current_state["target_weekday"]
        today_df = pd.read_json(StringIO(current_state["today_df_json"]), orient='split', convert_dates=['日期'])
        today_df['日期'] = pd.to_datetime(today_df['日期']).dt.date

        gsheets_g = gc.open_by_url(SHEET_URL)
        try: wk = gsheets_g.worksheet(SHEET_WEEKLY)
        except gspread.exceptions.WorksheetNotFound: wk = gsheets_g.add_worksheet(title=SHEET_WEEKLY, rows=100, cols=20)

        now_str = dt.now(TW_TZ).strftime("%Y-%m-%d %H:%M")
        row = {
            "週次起訖": f"{this_mon} ~ {this_sun}", "查詢星期": f"星期{target_weekday}",
            "當日課程列表": summarize_courses(today_df), "行前提醒一句話": edited_text,
            "更新時間": now_str
        }

        existing_records = wk.get_all_records()
        existing = pd.DataFrame(existing_records)
        outdf = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
        wk.clear()
        set_with_dataframe(wk, outdf, resize=True)

        return f"✅ 成功回存 星期 {target_weekday} 的提醒！(目前共 {len(outdf)} 筆紀錄)"
    except Exception as e:
        return f"❌ 儲存時發生錯誤：{e}"

# (*** 重大升級函式 ***)
def find_learning_resources(course_name, current_state):
    """
    Gradio 按下「幫我找資源」按鈕時執行的函式
    (新：會讀取 current_state 來取得課程內容)
    """
    if not course_name:
        return "請先選擇一門課程。"
    if not GEMINI_MODEL:
        return "❌ Gemini 模型未成功初始化，請檢查 Colab 輸出。"
    if not current_state:
        return "❌ 錯誤：請先「查詢課表」取得課程資料。"

    # 顯示「思考中」訊息
    yield "🧠 Gemini 思考中...請稍候..."

    # --- 1. 從 State 還原當日課表 ---
    try:
        today_df = pd.read_json(StringIO(current_state["today_df_json"]), orient='split', convert_dates=['日期'])
        today_df['日期'] = pd.to_datetime(today_df['日期']).dt.date

        # --- 2. 抓取該課程的「備註」與「先讀」---
        # 找到使用者選的那堂課
        course_info = today_df[today_df['課程名稱'] == course_name].iloc[0]

        notes = str(course_info.get('備註', ''))
        reading = str(course_info.get('先讀章節', ''))

    except Exception as e:
        # 如果出錯，就退回原本的廣泛搜尋
        notes = ""
        reading = ""
        print(f"⚠️ 讀取課程備註失敗 ({e})，將僅使用課程名稱搜尋。")

    # --- 3. 建立一個更豐富的 Prompt ---
    prompt = f"""
    你是一位專業的學習導師與 YouTube 影片搜集專家。
    我的大學課程是「{course_name}」。

    """

    # (新) 動態加入課程內容
    if notes or reading:
        prompt += "**請你「優先」針對以下我提供的「本週具體學習內容」進行推薦：**\n"
        if notes:
            prompt += f"* **本週主題 (來自我的備註)：** {notes}\n"
        if reading:
            prompt += f"* **指定閱讀：** {reading}\n"
        prompt += "\n"
    else:
        prompt += "請幫我推薦這門課的「入門」與「核心概念」相關資源。\n\n"


    prompt += """
    請幫我推薦 3 個 **繁體中文** 的 YouTube 教學影片（必須是高品質、解說清晰的頻道）
    以及 3 篇 **繁體中文** 的延伸閱讀文章（適合初學者的部落格或教學網站）。

    請嚴格使用以下 Markdown 格式回覆，並包含簡短說明：

    ### 🎥 YouTube 影片
    * [影片標題](URL) - 簡短說明 (約 15 字)
    * [影片標題](URL) - 簡短說明 (約 15 字)
    * [影片標題](URL) - 簡短說明 (約 15 字)

    ### 📚 延伸閱讀
    * [文章標題](URL) - 簡短說明 (約 15 字)
    * [文章標題](URL) - 簡短說明 (約 15 字)
    * [文章標題](URL) - 簡短說明 (約 15 字)
    """

    # (新) 印出我們送給 AI 的完整指令，方便除錯
    print("--- 傳送給 Gemini 的 Prompt ---")
    print(prompt)
    print("----------------------------")

    try:
        response = GEMINI_MODEL.generate_content(prompt)
        yield response.text
    except Exception as e:
        yield f"❌ 呼叫 Gemini API 時發生錯誤：{e}"

# === 4. 啟動 Gradio 介面 ===
print("Gradio 介面啟動中...")

with gr.Blocks(title="課表查詢與提醒編輯器") as demo:
    gr.Markdown("## 🗓️ 課表查詢與 AI 學習助理")
    gr.Markdown("請選擇要查詢的星期，系統將自動抓取「本週」的課表，並可推薦學習資源。")

    query_data_state = gr.State(value={})

    with gr.Row():
        weekday_selector = gr.Radio(
            ["一", "二", "三", "四", "五", "六", "日"],
            label="選擇星期",
            value="三"
        )
        query_btn = gr.Button("🔍 查詢當週課表", variant="primary")

    gr.Markdown("---")

    date_range_display = gr.Markdown("### 尚未查詢")
    course_display = gr.DataFrame(
        label="當日課程（請在 Google Sheet 填寫「先讀章節」或「備註」）",
        headers=["課程名稱","時間(起)","時間(迄)","地點","先讀章節","備註"]
    )

    gr.Markdown("### ✏️ 行前提醒（可編輯並回存）")
    with gr.Row():
        reminder_box = gr.Textbox(label="行前提醒一句話", lines=3, interactive=True, scale=3)
        save_btn = gr.Button("💾 提交並回存", scale=1)
    status = gr.Markdown()

    gr.Markdown("---")
    ai_title = gr.Markdown("## 💡 AI 學習資源推薦", visible=False)
    course_selector_dropdown = gr.Dropdown(
        label="選擇一門課（AI 會讀取該課程的「備註」欄）",
        choices=[],
        visible=False
    )
    find_resources_btn = gr.Button("🧠 幫我找資源 (呼叫 Gemini)", visible=False, variant="primary")
    resources_output = gr.Markdown()

    # --- 連結按鈕功能 ---

    # 1. 「查詢」按鈕
    query_btn.click(
        fn=query_schedule_and_reminder,
        inputs=[weekday_selector],
        outputs=[date_range_display,
                 course_display,
                 reminder_box,
                 query_data_state,
                 course_selector_dropdown,
                 find_resources_btn,
                 ai_title,
                 resources_output
                ]
    )

    # 2. 「儲存」按鈕
    save_btn.click(
        fn=save_edited_reminder,
        inputs=[reminder_box, query_data_state],
        outputs=[status]
    )

    # 3. (*** 重大升級 ***) 「幫我找資源」按鈕
    find_resources_btn.click(
        fn=find_learning_resources,
        # (新) 傳入下拉選單和「隱藏的狀態」
        inputs=[course_selector_dropdown, query_data_state],
        outputs=[resources_output]
    )

demo.launch(share=True, inline=True, debug=True)

正在從 Google Sheet 'HW6' 預載課表資料...
✅ 成功預載 192 筆課表資料。
✅ 成功初始化 Gemini Flash 模型。
Gradio 介面啟動中...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b6d47199aef6bb1b86.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


--- 傳送給 Gemini 的 Prompt ---

    你是一位專業的學習導師與 YouTube 影片搜集專家。
    我的大學課程是「電腦輔助製圖」。

    **請你「優先」針對以下我提供的「本週具體學習內容」進行推薦：**
* **本週主題 (來自我的備註)：** W11: 鈑金 (Sheet Metal)
* **指定閱讀：** SW Ch. 5


    請幫我推薦 3 個 **繁體中文** 的 YouTube 教學影片（必須是高品質、解說清晰的頻道）
    以及 3 篇 **繁體中文** 的延伸閱讀文章（適合初學者的部落格或教學網站）。

    請嚴格使用以下 Markdown 格式回覆，並包含簡短說明：

    ### 🎥 YouTube 影片
    * [影片標題](URL) - 簡短說明 (約 15 字)
    * [影片標題](URL) - 簡短說明 (約 15 字)
    * [影片標題](URL) - 簡短說明 (約 15 字)

    ### 📚 延伸閱讀
    * [文章標題](URL) - 簡短說明 (約 15 字)
    * [文章標題](URL) - 簡短說明 (約 15 字)
    * [文章標題](URL) - 簡短說明 (約 15 字)
    
----------------------------
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7863 <> https://b6d47199aef6bb1b86.gradio.live
